In [ ]:
import mlflow
import argparse
import os
import time
import joblib
import mlflow
import mlflow.sklearn
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
import xgboost as xgb
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
from sklearn.metrics import classification_report


os.chdir("..")  # move to project root
print("Current directory:", os.getcwd())

Current directory: c:\Users\Administrator.DESKTOP-P34765Q\Desktop\projects\credit-default-aml


In [3]:
df = pd.read_csv("./data/default_of_credit_card_clients.csv")
print(df.iloc[:, -1].value_counts())


Y
0                             23364
1                              6636
default payment next month        1
Name: count, dtype: int64


In [4]:
df.shape

(30001, 25)

In [ ]:
df.info()

In [6]:
import os

train_src_dir = "../src"
os.makedirs(train_src_dir, exist_ok=True) # create the dir if it doesn't exists

Next script writes/create the main() logic and the file 

In [ ]:
%%writefile {train_src_dir}/main_run.py


def main():
    """Main function of the script."""

    # input and output arguments
    parser = argparse.ArgumentParser()
    parser.add_argument("--data", type=str, help="path to input data")
    parser.add_argument("--test_train_ratio", type=float, required=False, default=0.25)
    parser.add_argument("--n_estimators", required=False, default=100, type=int)
    parser.add_argument("--learning_rate", required=False, default=0.1, type=float)
    parser.add_argument("--registered_model_name", type=str, help="model name")
    parser.add_argument("--saved_model_path", type=str, default="../models", help="path to save the model locally")
    args = parser.parse_args()
   
    # Start Logging
    mlflow.start_run()

    # enable autologging
    mlflow.sklearn.autolog()

    ###################
    #<prepare the data>
    ###################
    print(" ".join(f"{k}={v}" for k, v in vars(args).items()))

    print("input data:", args.data)
    
    credit_df = pd.read_csv(args.data, header=1, index_col=0)

    mlflow.log_metric("num_samples", credit_df.shape[0])
    mlflow.log_metric("num_features", credit_df.shape[1] - 1)

    train_df, test_df = train_test_split(
        credit_df,
        test_size=args.test_train_ratio,
    )
    ####################
    #</prepare the data>
    ####################

    ##################
    #<train the model>
    ##################
    # Extracting the label column
    y_train = train_df.pop("default payment next month")

    # convert the dataframe values to array
    X_train = train_df.values

    # Extracting the label column
    y_test = test_df.pop("default payment next month")

    # convert the dataframe values to array
    X_test = test_df.values

    print(f"Training with data of shape {X_train.shape}")

    clf = GradientBoostingClassifier(
        n_estimators=args.n_estimators, learning_rate=args.learning_rate
    )
    clf.fit(X_train, y_train)

    y_pred = clf.predict(X_test)

    print(classification_report(y_test, y_pred))
    ###################
    #</train the model>
    ###################

    ##########################
    #<save and register model>
    ##########################
    # Registering the model to the workspace
    print("Registering the model via MLFlow")

    mlflow.set_experiment("credit_default_aml_model")
    mlflow.register_model(
    "runs:/<RUN_ID>/model",
    "CreditDefaultModel"
)

    mlflow.sklearn.log_model(
        sk_model=clf,
        registered_model_name=args.registered_model_name,
        name=args.,
    )

    # Saving the model to a file
    mlflow.sklearn.save_model(
        sk_model=clf,
        path=os.path.join(args.saved_model_path, "trained_model"),
    )
    ###########################
    #</save and register model>
    ###########################
    
    # Stop Logging
    mlflow.end_run()

if __name__ == "__main__":
    main()

Overwriting ../src/main_run.py


In [8]:
!python src/main_run.py --data data/default_of_credit_card_clients.csv --registered_model_name gbc_local_model


data=data/default_of_credit_card_clients.csv test_train_ratio=0.25 n_estimators=100 learning_rate=0.1 registered_model_name=gbc_local_model
input data: data/default_of_credit_card_clients.csv
Training with data of shape (22500, 23)
              precision    recall  f1-score   support

           0       0.84      0.95      0.89      5874
           1       0.67      0.36      0.47      1626

    accuracy                           0.82      7500
   macro avg       0.75      0.66      0.68      7500
weighted avg       0.80      0.82      0.80      7500

Registering the model via MLFlow


2025/11/07 13:31:38 WARNING mlflow.sklearn: Failed to log training dataset information to MLflow Tracking. Reason: 'Series' object has no attribute 'flatten'
2025/11/07 13:36:39 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.
Successfully registered model 'gbc_local_model'.
Created version '1' of model 'gbc_local_model'.
